In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset"
)

df = dataset["train"].to_pandas()

In [2]:
import re
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Clean empty rows and group repeated instructions
# We’ll keep repeated instructions in the same split to prevent direct leakage

data = df.dropna(subset=["instruction", "response", "intent"]).copy()

for column in ["instruction", "response"]:
    data[column] = data[column].astype(str).str.strip()

data = data[
    data["instruction"].ne("") & data["response"].ne("")
].copy()

data = data.drop_duplicates(subset=["instruction", "response"])


def instruction_key(text):
    text = text.casefold()
    text = re.sub(r"\{\{.*?\}\}", " ENTITY ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


data["instruction_group"] = data["instruction"].map(instruction_key)

In [3]:
# Create three separate splits
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, heldout_idx = next(
    splitter.split(data, groups=data["instruction_group"])
)

train_pool = data.iloc[train_idx].copy()
heldout = data.iloc[heldout_idx].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=43,
)

validation_idx, test_idx = next(
    splitter.split(heldout, groups=heldout["instruction_group"])
)

validation_pool = heldout.iloc[validation_idx].copy()
test_pool = heldout.iloc[test_idx].copy()

In [4]:
# Select a small sample across intents
def sample_per_intent(frame, limit, seed):
    return pd.concat(
        [
            group.sample(n=min(limit, len(group)), random_state=seed)
            for _, group in frame.groupby("intent", sort=True)
        ],
        ignore_index=True,
    )


train = sample_per_intent(train_pool, limit=44, seed=42)
validation = sample_per_intent(validation_pool, limit=5, seed=43)
test = sample_per_intent(test_pool, limit=5, seed=44)

for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    print(name, "rows:", len(frame), "intents:", frame["intent"].nunique())

assert set(train.instruction_group).isdisjoint(validation.instruction_group)
assert set(train.instruction_group).isdisjoint(test.instruction_group)
assert set(validation.instruction_group).isdisjoint(test.instruction_group)

train rows: 2024 intents: 46
validation rows: 230 intents: 46
test rows: 230 intents: 46


In [5]:
# Inspect the entity names in the training sample
pattern = r"\{\{\s*([^{}]+?)\s*\}\}"

def entity_counts(frame, column):
    return (
        frame[column]
        .str.findall(pattern)
        .explode()
        .dropna()
        .str.strip()
        .value_counts()
    )


entities = (
    pd.concat(
        [
            entity_counts(train, "instruction").rename("in_instruction"),
            entity_counts(train, "response").rename("in_response"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

entities["total"] = entities.sum(axis=1)
entities = entities.sort_values("total", ascending=False)

print("Distinct entity names:", len(entities))
print(entities.head(40).to_string())

Distinct entity names: 533
                                in_instruction  in_response  total
Company Website URL                          0          586    586
Customer Support Phone Number                0          530    530
Order Number                                 0          423    423
Customer Support Working Hours               0          242    242
Order History                                0          199    199
Customer Support Team                        0          147    147
Email Address                                0           83     83
My Orders                                    0           81     81
Customer Support Email Address               0           69     69
Company Website                              0           56     56
Store Location                               0           53     53
Customer Support                             0           52     52
Account Settings                             0           46     46
Order Section                      

In [6]:
# Define the replacements
import json
import re
from collections import Counter
from pathlib import Path

ENTITY_PATTERN = re.compile(r"\{\{\s*([^{}]+?)\s*\}\}")


def normalize_entity(name):
    return " ".join(name.casefold().split())


# Fictional business information, also included in the model's input.
STORE_PROFILE = {
    "company_name": "Demo Retail",
    "website": "https://shop.example.com",
    "support_email": "support@example.com",
    "support_phone": "+1-202-555-0147",
    "support_hours": "Monday–Friday, 09:00–17:00 Eastern Time",
}

BUSINESS_REPLACEMENTS = {
    "Company Name": STORE_PROFILE["company_name"],
    "Company Website URL": STORE_PROFILE["website"],
    "Company Website": STORE_PROFILE["website"],
    "Website URL": STORE_PROFILE["website"],
    "Website": STORE_PROFILE["website"],
    "Customer Support Email": STORE_PROFILE["support_email"],
    "Customer Support Email Address": STORE_PROFILE["support_email"],
    "Customer Support Phone Number": STORE_PROFILE["support_phone"],
    "Customer Support Working Hours": STORE_PROFILE["support_hours"],
    "Customer Support Hours": STORE_PROFILE["support_hours"],
}

# Labels rather than customer-specific values.
INTERFACE_LABELS = [
    "Order History",
    "My Orders",
    "Account Settings",
    "Contact Us",
    "Live Chat",
    "Profile",
    "Returns",
    "Edit Order",
    "Exchange & Returns",
    "Refunds",
    "Modify Order",
    "Account",
    "Return Policy",
    "Purchase History",
    "App Store",
    "Google Play Store",
]

REPLACEMENTS = {
    normalize_entity(name): value
    for name, value in BUSINESS_REPLACEMENTS.items()
}

REPLACEMENTS.update({
    normalize_entity(name): name
    for name in INTERFACE_LABELS
})

REPLACEMENTS.update({
    "customer support team": "customer support team",
    "customer support": "customer support",
    "support": "support",
})

In [7]:
# Convert supported examples into chat format
SYSTEM_MESSAGE = (
    "You are a customer-support assistant for Demo Retail. "
    "Help users with retail questions. "
    "Use the supplied store details when relevant. "
    "You have not executed any actions or retrieved customer records. "
    "Do not claim that you have changed an order or checked its status.\n\n"
    "Store details:\n"
    + json.dumps(STORE_PROFILE, ensure_ascii=False, indent=2)
    + "\n\nInterface labels that may be referenced:\n"
    + ", ".join(INTERFACE_LABELS)
)


def replace_entities(text):
    return ENTITY_PATTERN.sub(
        lambda match: REPLACEMENTS[
            normalize_entity(match.group(1))
        ],
        text,
    )


def prepare_split(frame):
    records = []
    rejected_rows = []
    unknown_counts = Counter()

    for row in frame.itertuples(index=False):
        combined = row.instruction + "\n" + row.response

        names = {
            normalize_entity(name)
            for name in ENTITY_PATTERN.findall(combined)
        }

        unknown = names - REPLACEMENTS.keys()

        if unknown:
            unknown_counts.update(unknown)
            rejected_rows.append({
                "instruction": row.instruction,
                "intent": row.intent,
                "reason": "unresolved entities",
                "entities": sorted(unknown),
            })
            continue

        instruction = replace_entities(row.instruction)
        response = replace_entities(row.response)

        # Catch malformed or unmatched placeholder delimiters too.
        if any(
            marker in instruction or marker in response
            for marker in ["{{", "}}"]
        ):
            rejected_rows.append({
                "instruction": row.instruction,
                "intent": row.intent,
                "reason": "unmatched placeholder delimiters",
                "entities": [],
            })
            continue

        records.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": instruction},
            ],
            "completion": [
                {"role": "assistant", "content": response},
            ],
            "intent": row.intent,
            "instruction_group": row.instruction_group,
        })

    return records, rejected_rows, unknown_counts

In [8]:
# Prepare and inspect the result
prepared = {}
rejected = {}

for name, frame in {
    "train": train,
    "validation": validation,
    "test": test,
}.items():
    records, skipped, unknown = prepare_split(frame)

    prepared[name] = records
    rejected[name] = skipped

    print(
        f"{name}: kept {len(records)} / {len(frame)}, "
        f"intents retained: {len({r['intent'] for r in records})}"
    )

    if name == "train":
        print("Most common unresolved entities:")
        print(unknown.most_common(15))

for record in prepared["train"][:5]:
    print("\nUSER:", record["prompt"][-1]["content"])
    print("ASSISTANT:", record["completion"][0]["content"])

train: kept 1090 / 2024, intents retained: 46
Most common unresolved entities:
[('order number', 399), ('email address', 83), ('store location', 52), ('order section', 40), ('password', 35), ('name', 29), ('tracking number', 27), ('product name', 23), ('username', 21), ('app name', 20), ('order status', 19), ('return', 19), ('track order', 18), ('my purchases', 18), ('customer support channel', 17)]
validation: kept 134 / 230, intents retained: 40
test: kept 132 / 230, intents retained: 42

USER: wana add a fucking item to the cart help me
ASSISTANT: I'm sorry to hear that you're having trouble adding an item to your cart. I'll do my best to assist you. To add an item to your cart, please follow these steps:

1. Browse our website and find the item you want to purchase.
2. Click on the item to view its details.
3. Look for an "Add to Cart" button or a similar option and click on it.
4. You should see a message confirming that the item has been added to your cart.
5. Repeat the process 

In [9]:
# Export JSONL files into your Agent1 project
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "agent1").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("Cannot locate the Agent1 project root.")

output_dir = project_root / "data" / "finetuning" / "retail_pilot_v1"
output_dir.mkdir(parents=True, exist_ok=True)


def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


for name in ["train", "validation", "test"]:
    if not prepared[name]:
        raise ValueError(f"No usable examples remain in {name}.")

    write_jsonl(output_dir / f"{name}.jsonl", prepared[name])
    write_jsonl(output_dir / f"{name}_rejected.jsonl", rejected[name])

print("Exported to:", output_dir)


Exported to: d:\Projects\projects\agent1\data\finetuning\retail_pilot_v1
